# Inverse covariance estimation with MFCF-LoGo.

## Generate data

In [1]:
import numpy as np
from scipy import linalg

def generate_spd_precision(n=5, density=0.2, eps=1e-8, prng=None):
    A = prng.random(size=(n, n)) * 1000
    precision = (A + A.T) / 2.0  # make symmetric

    # Sparsify (keep zeros symmetric; don't zero the diagonal)
    mask = prng.uniform(size=(n, n)) < density
    mask = np.triu(mask, k=1)  # keep strictly upper triangle
    mask = mask + mask.T
    np.fill_diagonal(mask, False)
    precision[mask] = 0.0

    # Diagonal loading: preserve ALL off-diagonal zeros, ensure PD
    min_eig = np.linalg.eigvalsh(precision).min()
    if min_eig <= eps:
        precision += (-min_eig + eps) * np.eye(n)

    return precision

n_samples = 100
n_features = 1000
prng = np.random.RandomState(1)

prec = generate_spd_precision(n_features, density=0.8, prng=prng)
cov = linalg.inv(prec)
d = np.sqrt(np.diag(cov))
cov /= d
cov /= d[:, np.newaxis]
prec *= d
prec *= d[:, np.newaxis]

X = prng.multivariate_normal(np.zeros(n_features), cov, size=n_samples)

## Estimate the covariance and precision matrices

In [14]:
from sklearn.covariance import GraphicalLassoCV
from mfcf_logo import MFCFLoGoCV, MFCFLoGo
import time

emp_cov = np.dot(X.T, X) / n_samples

start = time.time()
model = MFCFLoGo(
    similarity="mutual_information",
    mi_n_neighbors=3,         # KSG neighbours, robust around 3-6
    mi_normalize="linfoot",   # default; sqrt(1 - exp(-2 I)) in [0, 1]
    mi_n_jobs=-1,             # forwarded to sklearn (>= 1.5)
)
model.fit(X)
end = time.time()
print(end - start)
cov_ = model.covariance_
prec_ = model.precision_

2.2898130416870117


In [15]:
prec_

array([[ 2.69771753e+07,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00, -6.99215383e+06],
       [ 0.00000000e+00,  3.45097107e+10,  1.02141451e+10, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  1.02141438e+10,  3.20687217e+10, ...,
         0.00000000e+00,  0.00000000e+00,  0.00000000e+00],
       ...,
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         4.41323136e+06,  0.00000000e+00,  0.00000000e+00],
       [ 0.00000000e+00,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  9.31719262e+06,  0.00000000e+00],
       [-6.99215383e+06,  0.00000000e+00,  0.00000000e+00, ...,
         0.00000000e+00,  0.00000000e+00,  5.25997591e+08]],
      shape=(1000, 1000))

In [16]:
start = time.time()
model = MFCFLoGo()
model.fit(X)
end = time.time()
print(end - start)
cov_ = model.covariance_
prec_ = model.precision_

0.16159820556640625


In [17]:
prec_

array([[7.81258293e+06, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 1.61813994e+09, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 3.45557326e+09, ...,
        0.00000000e+00, 0.00000000e+00, 0.00000000e+00],
       ...,
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        3.56211149e+06, 0.00000000e+00, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 9.06134927e+06, 0.00000000e+00],
       [0.00000000e+00, 0.00000000e+00, 0.00000000e+00, ...,
        0.00000000e+00, 0.00000000e+00, 6.38712675e+08]],
      shape=(1000, 1000))